# Gikomba Market Stall Performance Analysis

**Analyst:** Marion Oduor · PLP Data Analytics Cohort 2026

Wanjiku runs a fresh-produce stall at Gikomba Market. This notebook uses core Python only to answer her questions: how is the business doing, is it growing, and what should she restock?

In [1]:
daily_sales = [2300, 1950, 2800, 2100, 3600, 5200, 4100,
               2500, 2200, 3100, 2400, 3900, 5600, 4700]   # 14 days, KES
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"] * 2

# qty = kg in stock, reorder = reorder level (kg), cost/price = KES per kg
stock = {
    "Tomatoes": {"qty": 18, "reorder": 25, "cost": 60, "price": 90},
    "Onions":   {"qty": 40, "reorder": 20, "cost": 70, "price": 100},
    "Sukuma":   {"qty": 9,  "reorder": 15, "cost": 30, "price": 50},
    "Potatoes": {"qty": 55, "reorder": 30, "cost": 50, "price": 75},
    "Avocado":  {"qty": 12, "reorder": 10, "cost": 80, "price": 140},
}

## Task 1: Total, Average, Highest and Lowest Daily Sales

**Approach:** Use `sum()`, `len()`, `max()` and `min()` for the headline numbers. Then use `list.index()` to find the position of the best and worst days. The day name comes from the parallel `days` list, and the week number is `position // 7 + 1` because each week has 7 days.

In [2]:
total_sales = sum(daily_sales)
average_sales = total_sales / len(daily_sales)
highest_sales = max(daily_sales)
lowest_sales = min(daily_sales)

best_pos = daily_sales.index(highest_sales)
worst_pos = daily_sales.index(lowest_sales)

best_day, best_week = days[best_pos], best_pos // 7 + 1
worst_day, worst_week = days[worst_pos], worst_pos // 7 + 1

print(f"Total sales (14 days): KES {total_sales:,}")
print(f"Average daily sales:   KES {average_sales:,.2f}")
print(f"Highest day:           KES {highest_sales:,} ({best_day}, week {best_week})")
print(f"Lowest day:            KES {lowest_sales:,} ({worst_day}, week {worst_week})")

Total sales (14 days): KES 46,450
Average daily sales:   KES 3,317.86
Highest day:           KES 5,600 (Sat, week 2)
Lowest day:            KES 1,950 (Tue, week 1)


## Task 2: Weekly Totals and Week-on-Week Growth

**Approach:** The function `week_totals()` slices the list into week 1 (`sales[:7]`) and week 2 (`sales[7:14]`) and returns both totals as a tuple. The result is unpacked into two variables. Growth is `(week 2 - week 1) / week 1 * 100`.

In [3]:
def week_totals(sales):
    """Return a tuple (week 1 total, week 2 total) from a list of 14 daily sales."""
    week1_total = sum(sales[:7])
    week2_total = sum(sales[7:14])
    return week1_total, week2_total


week1_total, week2_total = week_totals(daily_sales)
growth_pct = (week2_total - week1_total) / week1_total * 100

print(f"Week 1 total: KES {week1_total:,}")
print(f"Week 2 total: KES {week2_total:,}")
print(f"Week-on-week growth: {growth_pct:.1f}%")

Week 1 total: KES 22,050
Week 2 total: KES 24,400
Week-on-week growth: 10.7%


## Task 3: Classifying Each Day as Strong, Normal or Weak

**Approach:** `classify_day()` compares a day's sales with the 14-day average. A day is **Strong** if it is more than 20% above average (`amount > average * 1.2`) and **Weak** if it is more than 20% below (`amount < average * 0.8`). Everything else is **Normal**. The function is applied to every day in a loop, and the dictionary counting pattern `counts.get(label, 0) + 1` tallies each category.

In [4]:
def classify_day(amount, average):
    """Return "Strong" if amount is more than 20% above average,
    "Weak" if more than 20% below average, otherwise "Normal"."""
    if amount > average * 1.2:
        return "Strong"
    elif amount < average * 0.8:
        return "Weak"
    else:
        return "Normal"


day_labels = []
day_counts = {}
for i, amount in enumerate(daily_sales):
    label = classify_day(amount, average_sales)
    day_labels.append(label)
    day_counts[label] = day_counts.get(label, 0) + 1
    week_no = i // 7 + 1
    print(f"Week {week_no} {days[i]}: KES {amount:>5,} -> {label}")

print()
print("Day counts:", day_counts)

Week 1 Mon: KES 2,300 -> Weak
Week 1 Tue: KES 1,950 -> Weak
Week 1 Wed: KES 2,800 -> Normal
Week 1 Thu: KES 2,100 -> Weak
Week 1 Fri: KES 3,600 -> Normal
Week 1 Sat: KES 5,200 -> Strong
Week 1 Sun: KES 4,100 -> Strong
Week 2 Mon: KES 2,500 -> Weak
Week 2 Tue: KES 2,200 -> Weak
Week 2 Wed: KES 3,100 -> Normal
Week 2 Thu: KES 2,400 -> Weak
Week 2 Fri: KES 3,900 -> Normal
Week 2 Sat: KES 5,600 -> Strong
Week 2 Sun: KES 4,700 -> Strong

Day counts: {'Weak': 6, 'Normal': 4, 'Strong': 4}


## Task 4: Weekdays versus Weekends

**Approach:** Loop through the sales with `enumerate()` and use the matching day name to place each amount in either the weekday list (Mon to Fri) or the weekend list (Sat and Sun). The two averages are compared with `(weekend - weekday) / weekday * 100`.

In [5]:
weekday_sales = []
weekend_sales = []
for i, amount in enumerate(daily_sales):
    if days[i] in ("Sat", "Sun"):
        weekend_sales.append(amount)
    else:
        weekday_sales.append(amount)

weekday_avg = sum(weekday_sales) / len(weekday_sales)
weekend_avg = sum(weekend_sales) / len(weekend_sales)
weekend_uplift_pct = (weekend_avg - weekday_avg) / weekday_avg * 100

print(f"Weekday average (Mon-Fri): KES {weekday_avg:,.2f} over {len(weekday_sales)} days")
print(f"Weekend average (Sat-Sun): KES {weekend_avg:,.2f} over {len(weekend_sales)} days")
print(f"Weekends sell {weekend_uplift_pct:.1f}% more per day than weekdays")

Weekday average (Mon-Fri): KES 2,685.00 over 10 days
Weekend average (Sat-Sun): KES 4,900.00 over 4 days
Weekends sell 82.5% more per day than weekdays


## Task 5: Stock Check and Gross Margins

**Approach:** The function `products_to_reorder()` loops through the stock dictionary with `.items()` and collects the names where `qty <= reorder`. A second function, `gross_margin_pct()`, applies `(price - cost) / price * 100` and is called for every product, so the formula is written only once.

In [6]:
def products_to_reorder(stock_register):
    """Return a list of product names whose quantity is at or below the reorder level."""
    low_stock = []
    for name, info in stock_register.items():
        if info["qty"] <= info["reorder"]:
            low_stock.append(name)
    return low_stock


def gross_margin_pct(price, cost):
    """Return gross margin as a percentage of price: (price - cost) / price * 100."""
    return (price - cost) / price * 100


reorder_list = products_to_reorder(stock)
margins = {}
for name, info in stock.items():
    margins[name] = gross_margin_pct(info["price"], info["cost"])

print("Products to reorder:", reorder_list)
print()
for name, info in stock.items():
    gap = info["reorder"] - info["qty"]
    note = f"  <- {gap} kg below reorder level" if name in reorder_list else ""
    print(f"{name:<9} margin = {margins[name]:.1f}%{note}")

Products to reorder: ['Tomatoes', 'Sukuma']

Tomatoes  margin = 33.3%  <- 7 kg below reorder level
Onions    margin = 30.0%
Sukuma    margin = 40.0%  <- 6 kg below reorder level
Potatoes  margin = 33.3%
Avocado   margin = 42.9%


## Task 6: Stall Performance Report

**Approach:** All results from Tasks 1 to 5 are brought together in one report. F-string alignment (`:<`, `:>`) lines up the columns, and `:,` adds thousands separators to every KES amount.

In [7]:
line = "=" * 52
print(line)
print("STALL PERFORMANCE REPORT - Wanjiku, Gikomba Market")
print(line)

print("\nSALES SUMMARY (14 days)")
print(f"  {'Total sales':<24}KES {total_sales:>10,}")
print(f"  {'Average per day':<24}KES {average_sales:>10,.2f}")
print(f"  {'Best day':<24}KES {highest_sales:>10,}  ({best_day}, week {best_week})")
print(f"  {'Worst day':<24}KES {lowest_sales:>10,}  ({worst_day}, week {worst_week})")

print("\nGROWTH")
print(f"  {'Week 1 total':<24}KES {week1_total:>10,}")
print(f"  {'Week 2 total':<24}KES {week2_total:>10,}")
print(f"  {'Week-on-week growth':<24}{growth_pct:>14.1f}%")

print("\nDAY CLASSIFICATION")
for label in ("Strong", "Normal", "Weak"):
    print(f"  {label:<24}{day_counts.get(label, 0):>3} days")

print("\nWEEKDAY vs WEEKEND (average per day)")
print(f"  {'Weekday (Mon-Fri)':<24}KES {weekday_avg:>10,.2f}")
print(f"  {'Weekend (Sat-Sun)':<24}KES {weekend_avg:>10,.2f}")
print(f"  {'Weekend uplift':<24}{weekend_uplift_pct:>14.1f}%")

print("\nSTOCK AND MARGINS")
print(f"  {'Product':<10}{'Stock':>7}{'Reorder':>9}{'Margin':>9}  Status")
for name, info in stock.items():
    status = "REORDER" if name in reorder_list else "OK"
    print(f"  {name:<10}{info['qty']:>5}kg{info['reorder']:>7}kg{margins[name]:>8.1f}%  {status}")
print(f"\n  Restock now: {', '.join(reorder_list)}")
print(line)

STALL PERFORMANCE REPORT - Wanjiku, Gikomba Market

SALES SUMMARY (14 days)
  Total sales             KES     46,450
  Average per day         KES   3,317.86
  Best day                KES      5,600  (Sat, week 2)
  Worst day               KES      1,950  (Tue, week 1)

GROWTH
  Week 1 total            KES     22,050
  Week 2 total            KES     24,400
  Week-on-week growth               10.7%

DAY CLASSIFICATION
  Strong                    4 days
  Normal                    4 days
  Weak                      6 days

WEEKDAY vs WEEKEND (average per day)
  Weekday (Mon-Fri)       KES   2,685.00
  Weekend (Sat-Sun)       KES   4,900.00
  Weekend uplift                    82.5%

STOCK AND MARGINS
  Product     Stock  Reorder   Margin  Status
  Tomatoes     18kg     25kg    33.3%  REORDER
  Onions       40kg     20kg    30.0%  OK
  Sukuma        9kg     15kg    40.0%  REORDER
  Potatoes     55kg     30kg    33.3%  OK
  Avocado      12kg     10kg    42.9%  OK

  Restock now: Tomatoes, 

## Task 7: Business Recommendations

**Approach:** Each recommendation is backed by one number taken from Tasks 1 to 6.

## Recommendations for Wanjiku

Restock tomatoes (18 kg against a 25 kg reorder level) and sukuma (9 kg against 15 kg) now, before the weekend rush. Stock up for Saturdays and Sundays, which average KES 4,900 a day, or 82.5% more than the KES 2,685 weekday average. Promote avocados, your highest-margin product at 42.9%, and keep them above the 10 kg reorder level. Run a weekday offer such as a sukuma and tomato bundle, because 6 of the 14 days were Weak and the lowest was Tuesday at KES 1,950. Keep up the momentum, since sales grew 10.7% from KES 22,050 in week 1 to KES 24,400 in week 2.